In [1]:
# --- STEP 1: READ AND CHECK OUR FILE.TXT ---
file_name = 'test1_IPcheckCleantalk.txt'

with open(file_name, 'r', encoding='utf-8') as f:
    my_ips = [line.strip() for line in f if line.strip()]

print(f"Total IPs found: {len(my_ips)}")
print("-" * 30)

# 1. Show the Top 3
print("TOP 3 IPs:")
for ip in my_ips[:3]:
    print(f"  > {ip}")

print("-" * 30)

# 2. Show the Bottom 3
print("BOTTOM 3 IPs:")
for ip in my_ips[-3:]:
    print(f"  > {ip}")

Total IPs found: 2985
------------------------------
TOP 3 IPs:
  > 58.27.0.232
  > 58.27.0.241
  > 58.27.14.219
------------------------------
BOTTOM 3 IPs:
  > 2404:6800:4001:812::200a
  > 2a05:d018:91c:3200:2846:99fb:81b6:1e11
  > 2a05:d018:91c:3200:5e0d:21a9:26ca:90b5


In [4]:
# --- STEP 2: CHECK API CONNECTIVITY ---
import requests

# --- CONFIGURATION ---
CLEANTALK_KEY = 'uhe8e4u6e6una2e' # Your CleanTalk Access Key

def test_cleantalk_api():
    # CleanTalk API Endpoint
    url = 'https://api.cleantalk.org/'
    
    # Parameters required by CleanTalk
    params = {
        'method_name': 'spam_check',
        'auth_key': CLEANTALK_KEY,
        'ip': '8.8.8.8'  # Testing with Google DNS
    }
    
    # 1. Make the request
    response = requests.get(url, params=params)
    
    # 2. Check the HTTP status
    if response.status_code == 200:
        data = response.json()
        
        # 3. CleanTalk returns "error_message" even if status is 200 if key is wrong
        if 'error_message' in data:
            print(f"❌ CleanTalk Error: {data['error_message']}")
            return False
        
        print("✅ CleanTalk Success! Connection established.")
        return True
    else:
        print(f"❌ HTTP Failed: {response.status_code}")
        return False

# Run the test
api_works = test_cleantalk_api()

✅ CleanTalk Success! Connection established.


In [8]:
# --- STEP 3: GET TO KNOW API DATA (VARIABLE)---
import requests
import pprint

# --- CONFIGURATION ---
CLEANTALK_KEY = 'uhe8e4u6e6una2e'
IP_TO_CHECK = '8.8.8.8'

# 1. CleanTalk Endpoint
url = 'https://api.cleantalk.org/'

# 2. Setup the request body
payload = {
    'method_name': 'spam_check',
    'auth_key': CLEANTALK_KEY,
    'ip': IP_TO_CHECK
}

# 3. Request (using GET)
response = requests.get(url, params=payload)

# 4. Print results
if response.status_code == 200:
    print(f"--- RAW JSON DATA FOR {IP_TO_CHECK} ---")
    pprint.pprint(response.json())
else:
    print(f"Error {response.status_code}: {response.text}")

--- RAW JSON DATA FOR 8.8.8.8 ---
{'data': {'8.8.8.8': {'appears': 0,
                      'country': 'US',
                      'domains_count': 3471,
                      'domains_list': None,
                      'frequency': 8,
                      'in_antispam': 0,
                      'in_antispam_previous': 1,
                      'in_antispam_updated': '2026-02-17 08:49:02',
                      'in_offline': 0,
                      'in_security': 0,
                      'network_type': 'hosting',
                      'sha256': '838c4c2573848f58e74332341a7ca6bc5cd86a8aec7d644137d53b4d597f10f5',
                      'spam_frequency_24h': 0,
                      'spam_rate': 0,
                      'submitted': '2022-02-09 21:05:06',
                      'updated': '2026-02-21 04:02:36'}}}


In [6]:
# --- STEP 4: FETCH AND STORE EVERY GITHUB SOURCE IN MASTER DATA.---
import sys
import requests

# --- CONFIGURATION ---
CLEANTALK_KEY = 'uhe8e4u6e6una2e'
MASTER_DATA = []

if api_works:
    # Check the first 1000 for now
    test_list = my_ips[:3000] 
    
    for index, ip in enumerate(test_list):
        # Progress Counter
        sys.stdout.write(f"\r🔍 Checking: {index + 1} of {len(test_list)}... ({ip})")
        sys.stdout.flush()

        url = 'https://api.cleantalk.org/'
        params = {
            'method_name': 'spam_check',
            'auth_key': CLEANTALK_KEY,
            'ip': ip
        }
        
        try:
            # CleanTalk uses GET with params
            response = requests.get(url, params=params)
            res_json = response.json()
            
            # Extract the specific IP data from the nested CleanTalk response
            # CleanTalk returns: {"data": {"8.8.8.8": {...}}}
            if 'data' in res_json and ip in res_json['data']:
                ip_data = res_json['data'][ip]
                ip_data['ip'] = ip  # Make sure the IP is inside the dict for Excel
                MASTER_DATA.append(ip_data)
        except:
            continue

print("\n\n✅ Done! Data collected for Excel.")

🔍 Checking: 2985 of 2985... (2a05:d018:91c:3200:5e0d:21a9:26ca:90b5))

✅ Done! Data collected for Excel.


In [7]:
# --- STEP 5: PRINT OUT THE API DATA (VARIABLE) AND ADJUST ACCORDING TO COLUMN. DOWNLOAD FILE.XLSX ---
import pandas as pd

# 1. Check if we have data
if not MASTER_DATA:
    print("No data found! Please run the loop first.")
else:
    # 2. Convert to Table
    df = pd.DataFrame(MASTER_DATA)

    # UPDATED COLUMNS based on your real JSON response
    columns_to_keep = {
        'ip': 'IP Address',
        'country': 'Country',
        'network_type': 'Network Type',
        'frequency': 'Spam Frequency',
	'sha256': 'IP Hash (SHA256)',
        'updated': 'Last Updated',
        'spam_rate': 'Spam Probability',
        'appears': 'On Blacklist (1=Yes)',
       
    }
    
    # Filter only what actually exists in your data
    existing = [c for c in columns_to_keep.keys() if c in df.columns]
    df_final = df[existing].rename(columns=columns_to_keep)

    df_final.to_excel("CleanTalk_Report.xlsx", index=False)
    print("\n✅ Excel Created with real CleanTalk data!")


✅ Excel Created with real CleanTalk data!
